In [ ]:

import requests

# HAPI FHIR R4 API endpoint
base_url = "https://hapi.fhir.org/baseR4"

# Patient identifier
patient_id = "nhanes-112200"

# API request URL
url = f"{base_url}/Observation"

# Search parameters
params = {
    "patient": patient_id,
    "_count": 20,
    "_format": "json"
}

# Send a GET request
response = requests.get(url, params=params, timeout=30)

# Display the HTTP status code
print("HTTP Status Code:", response.status_code)

# Convert the response into JSON
response.raise_for_status()
data = response.json()

# Display the number of observations on the first page
print("Records on first page:", len(data.get("entry", [])))

# Display whether another page is available
print("Additional pages available:",
      any(link.get("relation") == "next"
          for link in data.get("link", [])))


In [ ]:

# Store all observations
all_observations = []

# Start with the first page URL
next_url = response.url

# Retrieve every available page
while next_url:
    page_response = requests.get(
        next_url,
        headers={"Accept": "application/fhir+json"},
        timeout=30
    )
    page_response.raise_for_status()

    page_data = page_response.json()

    # Extract Observation resources
    for entry in page_data.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") == "Observation":
            all_observations.append(resource)

    # Find the next page URL
    next_url = next(
        (
            link["url"]
            for link in page_data.get("link", [])
            if link.get("relation") == "next"
        ),
        None
    )

# Display results
print("Total Observation records retrieved:",
      len(all_observations))


In [ ]:

import pandas as pd

# Create a list to store structured records
observation_data = []

for obs in all_observations:

    # Extract patient reference
    patient_reference = obs.get("subject", {}).get("reference")

    # Extract observation code and description
    coding = obs.get("code", {}).get("coding", [])
    code = coding[0].get("code") if coding else None
    description = obs.get("code", {}).get("text")

    if not description and coding:
        description = coding[0].get("display")

    # Extract numeric measurement, if available
    value = obs.get("valueQuantity", {})

    observation_data.append({
        "observation_id": obs.get("id"),
        "patient_id": patient_reference,
        "status": obs.get("status"),
        "loinc_code": code,
        "test_name": description,
        "value": value.get("value"),
        "unit": value.get("unit"),
        "date": obs.get("effectiveDateTime")
    })

# Convert into a DataFrame
df_observations = pd.DataFrame(observation_data)

# Display the first 5 rows
display(df_observations.head())

print("Total rows:", len(df_observations))


In [ ]:

# Check the shape of our dataset
print("Dataset shape:", df_observations.shape)

# Check for missing values in each column
print("\nMissing values by column:")
print(df_observations.isnull().sum())

# Check for duplicate Observation IDs
print("\nDuplicate Observation IDs:")
print(df_observations["observation_id"].duplicated().sum())

# Check data types
print("\nColumn data types:")
print(df_observations.dtypes)


In [ ]:

# Convert the date column to datetime format
df_observations["date"] = pd.to_datetime(
    df_observations["date"],
    errors="coerce",
    utc=True
)

# Check the updated data type
print("Updated date data type:")
print(df_observations["date"].dtype)

# Check whether any dates failed conversion
print("\nInvalid dates:")
print(df_observations["date"].isnull().sum())

# Display the first five records
display(df_observations.head())


In [ ]:

# Find observations with invalid dates
invalid_dates = df_observations[
    df_observations["date"].isna()
]

# Display the affected observation IDs
display(invalid_dates[
    ["observation_id", "test_name", "date"]
])


In [ ]:

# Find the original date values for the 8 affected records

invalid_ids = set(
    invalid_dates["observation_id"].astype(str)
)

for obs in all_observations:
    if str(obs.get("id")) in invalid_ids:
        print("Observation ID:", obs.get("id"))
        print("effectiveDateTime:", obs.get("effectiveDateTime"))
        print("effectivePeriod:", obs.get("effectivePeriod"))
        print("issued:", obs.get("issued"))


In [ ]:

# Recover the original dates from FHIR JSON
original_dates = {
    str(obs.get("id")): obs.get("effectiveDateTime")
    for obs in all_observations
}

# Rebuild the date column
df_observations["date"] = df_observations[
    "observation_id"
].astype(str).map(original_dates)

# Convert mixed date formats to datetime
df_observations["date"] = pd.to_datetime(
    df_observations["date"],
    format="mixed",
    errors="coerce",
    utc=True
)

# Verify the results
print("Total observations:", len(df_observations))
print("Invalid dates:", df_observations["date"].isna().sum())
print("Date data type:", df_observations["date"].dtype)


In [ ]:

# Count observations by laboratory test

test_summary = (
    df_observations
    .groupby(["loinc_code", "test_name"])
    .size()
    .reset_index(name="number_of_observations")
    .sort_values(
        by="number_of_observations",
        ascending=False
    )
)

# Display the results
display(test_summary)

print("Total unique LOINC codes:",
      df_observations["loinc_code"].nunique())

print("Total observations:",
      len(df_observations))


In [ ]:

# Check how many different patient references appear
print("Unique patient references:")
print(df_observations["patient_id"].value_counts())

# Verify all records reference our selected patient
expected_patient = "Patient/nhanes-112200"

mismatched_records = df_observations[
    df_observations["patient_id"] != expected_patient
]

print("\nRecords with mismatched patient references:",
      len(mismatched_records))

# Display any mismatched records
display(mismatched_records[
    ["observation_id", "patient_id", "test_name"]
])


In [ ]:

# Save the cleaned FHIR Observation dataset

df_observations.to_csv(
    "FHIR_Observations_Cleaned.csv",
    index=False
)

print("Dataset saved successfully!")
print("Total records:", len(df_observations))
print("Total columns:", len(df_observations.columns))


In [ ]:

import requests

# Test connection to HAPI FHIR
try:
    response_test = requests.get(
        "https://hapi.fhir.org/baseR4/metadata",
        timeout=20
    )

    print("HTTP Status Code:", response_test.status_code)

except requests.exceptions.RequestException as e:
    print("Connection failed")
    print("Error type:", type(e).__name__)
    print("Error:", str(e))


In [ ]:

# Retrieve 20 patient records from HAPI FHIR

patient_url = f"{base_url}/Patient"

params = {
    "_count": 20,
    "_format": "json"
}

patient_response = requests.get(
    patient_url,
    params=params,
    timeout=30
)

patient_response.raise_for_status()

patient_bundle = patient_response.json()

# Extract patient resources
patient_records = [
    entry["resource"]
    for entry in patient_bundle.get("entry", [])
    if entry.get("resource", {}).get("resourceType") == "Patient"
]

print("Total patients retrieved:", len(patient_records))

# Display the first 5 patient IDs
print("\nFirst 5 Patient IDs:")

for patient in patient_records[:5]:
    print(patient.get("id"))


In [ ]:

import pandas as pd
import requests
import time

patient_observation_counts = []

# Check Observation counts for all 20 patients
for patient in patient_records:
    patient_id = patient.get("id")

    try:
        result = requests.get(
            f"{base_url}/Observation",
            params={
                "patient": patient_id,
                "_summary": "count",
                "_format": "json"
            },
            timeout=30
        )

        result.raise_for_status()
        bundle = result.json()

        observation_count = bundle.get("total")

    except requests.exceptions.RequestException as e:
        print(f"Request failed for {patient_id}: {e}")
        observation_count = None

    patient_observation_counts.append({
        "patient_id": patient_id,
        "observation_count": observation_count
    })

    # Avoid sending requests too quickly
    time.sleep(0.5)

# Convert results into a DataFrame
df_patient_counts = pd.DataFrame(patient_observation_counts)

display(df_patient_counts)

print(
    "Patients with observations:",
    (df_patient_counts["observation_count"] > 0).sum()
)


In [ ]:

# Calculate total available observations

total_observations = df_patient_counts[
    "observation_count"
].sum()

print("Total patients:", len(df_patient_counts))

print(
    "Total available observations:",
    int(total_observations)
)

print(
    "Average observations per patient:",
    round(df_patient_counts["observation_count"].mean(), 2)
)

print(
    "Minimum observations:",
    df_patient_counts["observation_count"].min()
)

print(
    "Maximum observations:",
    df_patient_counts["observation_count"].max()
)


In [ ]:

import requests
import time

# Store all observations from the 20 patients
all_patient_observations = []

# Keep track of extraction results
extraction_results = []

for patient in patient_records:

    patient_id = patient.get("id")
    patient_observations = []

    # Start with the first page
    next_url = f"{base_url}/Observation"
    params = {
        "patient": patient_id,
        "_count": 50,
        "_format": "json"
    }

    # Retrieve all pages for this patient
    while next_url:

        response = requests.get(
            next_url,
            params=params,
            timeout=30
        )

        response.raise_for_status()
        bundle = response.json()

        # Extract and validate Observation resources
        for entry in bundle.get("entry", []):

            obs = entry.get("resource", {})

            if (
                obs.get("resourceType") == "Observation"
                and obs.get("subject", {}).get("reference")
                == f"Patient/{patient_id}"
            ):
                patient_observations.append(obs)

        # Find the next page
        next_url = next(
            (
                link["url"]
                for link in bundle.get("link", [])
                if link.get("relation") == "next"
            ),
            None
        )

        # Next-page URLs already include search parameters
        params = None

    # Combine this patient's observations
    all_patient_observations.extend(patient_observations)

    extraction_results.append({
        "patient_id": patient_id,
        "retrieved": len(patient_observations)
    })

    print(
        f"{patient_id}: "
        f"{len(patient_observations)} observations"
    )

    # Brief pause between patients
    time.sleep(0.5)

print("Total patients:", len(patient_records))
print(
    "Total observations retrieved:",
    len(all_patient_observations)
)


In [ ]:

import pandas as pd

# Convert extraction results into a DataFrame
df_extraction = pd.DataFrame(extraction_results)

# Check total records
print("Total observations retrieved:",
      len(all_patient_observations))

# Check duplicate Observation IDs
observation_ids = [
    obs.get("id")
    for obs in all_patient_observations
]

duplicate_count = (
    pd.Series(observation_ids).duplicated().sum()
)

print("Duplicate Observation IDs:", duplicate_count)

# Compare retrieved counts with expected counts
df_validation = df_extraction.merge(
    df_patient_counts,
    on="patient_id",
    how="left"
)

df_validation["difference"] = (
    df_validation["retrieved"] -
    df_validation["observation_count"]
)

# Display validation results
display(df_validation)

print(
    "Patients with count mismatches:",
    (df_validation["difference"] != 0).sum()
)


In [ ]:

import pandas as pd

# Store structured clinical observations
structured_observations = []

for obs in all_patient_observations:

    # Extract clinical test information
    coding = obs.get("code", {}).get("coding", [])
    first_code = coding[0] if coding else {}

    # Extract measurement values
    value_quantity = obs.get("valueQuantity", {})

    # Extract patient reference
    patient_reference = obs.get(
        "subject", {}
    ).get("reference")

    # Create a structured record
    structured_observations.append({
        "observation_id": obs.get("id"),
        "patient_id": patient_reference,
        "status": obs.get("status"),
        "loinc_code": first_code.get("code"),
        "test_name": (
            obs.get("code", {}).get("text")
            or first_code.get("display")
        ),
        "value": value_quantity.get("value"),
        "unit": value_quantity.get("unit"),
        "date_original": obs.get("effectiveDateTime")
    })

# Convert records into a DataFrame
df_all_observations = pd.DataFrame(
    structured_observations
)

# Display first 5 records
display(df_all_observations.head())

# Check dataset dimensions
print("Dataset shape:", df_all_observations.shape)
print("Unique patients:",
      df_all_observations["patient_id"].nunique())


In [ ]:

# STEP 35: Data Quality Assessment

print("DATASET OVERVIEW")

print("Total rows:", len(df_all_observations))
print("Total columns:", len(df_all_observations.columns))
print("Unique patients:", df_all_observations["patient_id"].nunique())

# Check missing values
print("\nMISSING VALUES BY COLUMN")

print(df_all_observations.isnull().sum())

# Check duplicate observation IDs
print("\nDUPLICATE OBSERVATION IDs")

print(df_all_observations["observation_id"].duplicated().sum())

# Check column data types
print("\nCOLUMN DATA TYPES")

print(df_all_observations.dtypes)

# Check measurement value ranges
print("\nMEASUREMENT VALUE SUMMARY")

print(df_all_observations["value"].describe())


In [ ]:

# STEP 36: Standardize FHIR Observation dates

# Preserve original dates
df_all_observations["date"] = pd.to_datetime(
    df_all_observations["date_original"],
    format="mixed",
    errors="coerce",
    utc=True
)

# Validate date conversion
print("Total observations:", len(df_all_observations))

print(
    "Successfully converted dates:",
    df_all_observations["date"].notna().sum()
)

print(
    "Invalid dates:",
    df_all_observations["date"].isna().sum()
)

print(
    "Date data type:",
    df_all_observations["date"].dtype
)

# Display sample records
display(
    df_all_observations[
        ["observation_id", "date_original", "date"]
    ].head()
)


In [ ]:

# STEP 37: Standardize Patient IDs

# Preserve the original FHIR reference
df_all_observations["patient_reference"] = (
    df_all_observations["patient_id"]
)

# Extract the patient identifier
df_all_observations["patient_id"] = (
    df_all_observations["patient_reference"]
    .str.replace(r"^Patient/", "", regex=True)
)

# Verify the results
print("Total observations:", len(df_all_observations))

print(
    "Unique patient IDs:",
    df_all_observations["patient_id"].nunique()
)

print(
    "Missing patient IDs:",
    df_all_observations["patient_id"].isna().sum()
)

# Compare original references with clean IDs
display(
    df_all_observations[
        ["patient_reference", "patient_id"]
    ].drop_duplicates().head()
)


In [ ]:

# STEP 38: Create the Patients table

import pandas as pd

patient_data = []

for patient in patient_records:

    # Extract patient name
    names = patient.get("name", [])
    first_name = None
    last_name = None

    if names:
        first_name = " ".join(names[0].get("given", []))
        last_name = names[0].get("family")

    # Create a structured patient record
    patient_data.append({
        "patient_id": patient.get("id"),
        "first_name": first_name,
        "last_name": last_name,
        "gender": patient.get("gender"),
        "birth_date": patient.get("birthDate"),
        "active": patient.get("active")
    })

# Convert into a DataFrame
df_patients = pd.DataFrame(patient_data)

# Display the first five patients
display(df_patients.head())

# Validate the patient table
print("Total patient records:", len(df_patients))

print(
    "Unique Patient IDs:",
    df_patients["patient_id"].nunique()
)

print(
    "Duplicate Patient IDs:",
    df_patients["patient_id"].duplicated().sum()
)

print("\nMissing values:")
print(df_patients.isnull().sum())


In [ ]:

# STEP 39: Validate Patient-Observation Relationships

# Identify all unique patient IDs
patient_ids = set(df_patients["patient_id"])

# Find observations referencing unknown patients
unmatched_observations = df_all_observations[
    ~df_all_observations["patient_id"].isin(patient_ids)
]

print("Total patients:", len(df_patients))

print("Total observations:", len(df_all_observations))

print(
    "Observations with unmatched Patient IDs:",
    len(unmatched_observations)
)

# Check how many patients have observations
patients_with_observations = (
    df_all_observations["patient_id"]
    .nunique()
)

print(
    "Patients with observations:",
    patients_with_observations
)

# Check relationship using a SQL-style join
df_joined = df_all_observations.merge(
    df_patients,
    on="patient_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nJoin validation:")
print(df_joined["_merge"].value_counts())

print("\nJoined dataset shape:", df_joined.shape)


In [ ]:

# STEP 40: Save structured FHIR datasets

# Save Patients table
df_patients.to_csv(
    "FHIR_Patients.csv",
    index=False
)

# Save Observations table
df_all_observations.to_csv(
    "FHIR_Observations.csv",
    index=False
)

print("Both datasets saved successfully!")

print("Patients:", len(df_patients))
print("Observations:", len(df_all_observations))


In [ ]:
import os

print(os.path.exists("FHIR_Patients.csv"))
print(os.path.exists("FHIR_Observations.csv"))

In [ ]:

# Check whether our DataFrames are still available

print("Patients DataFrame:",
      "df_patients" in globals())

print("Observations DataFrame:",
      "df_all_observations" in globals())


In [ ]:

# Convert Boolean values into MySQL-compatible integers
df_patients["active"] = df_patients["active"].map({
    True: 1,
    False: 0
})

# Save the corrected CSV
df_patients.to_csv(
    "FHIR_Patients_Fixed.csv",
    index=False
)

# Verify results
display(df_patients[["patient_id", "active"]].head())

print("Corrected CSV saved successfully!")


In [ ]:

#  Prepare Observations for MySQL

# Create a separate copy
df_observations_mysql = df_all_observations.copy()

# Convert timezone-aware dates to MySQL DATETIME strings
df_observations_mysql["date"] = (
    pd.to_datetime(
        df_observations_mysql["date"],
        utc=True
    )
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

# Save MySQL-compatible CSV
df_observations_mysql.to_csv(
    "FHIR_Observations_MySQL.csv",
    index=False
)

print("MySQL CSV created successfully!")
print("Total records:", len(df_observations_mysql))

# Verify date format
display(
    df_observations_mysql[
        ["observation_id", "date_original", "date"]
    ].head()
)
